# Streaming

> Streaming helpers for lossless event collation.

In [ ]:
#| default_exp streaming

## Imports

In [ ]:
#| export
import json
from dataclasses import dataclass, field, fields
from fastcore.utils import *
from fastcore.meta import delegates
from fastspec.errors import *
from fastllm.types import *
from fastllm.normalize import *

In [ ]:
#| hide
import yaml,json
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from cachy.core import enable_cachy, disable_cachy, doms

In [ ]:
enable_cachy(doms=(*doms,'api.moonshot.ai', 'chatgpt.com'), hdrs=('content-type',))

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
specs_path = Path('../specs/')

ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
kimi_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['MOONSHOT_API_KEY']}"})
for op in kimi_cli.ops: op.base_url = 'https://api.moonshot.ai/v1'

In [ ]:
ant_spec, oai_spec, gem_spec

(SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

In [ ]:
gem_cli.models

- models.generate_content(model, contents, system_instruction, tools, tool_config, safety_settings, generation_config, cached_content, service_tier, store): *Generates a model response given an input `GenerateContentRequest`. Refer to the [text generation guide](https://ai.google.dev/gemini-api/docs/text-generation) for detailed usage information. Input capabilities differ between models, including tuned models. Refer to the [model guide](https://ai.google.dev/gemini-api/docs/models/gemini) and [tuning guide](https://ai.google.dev/gemini-api/docs/model-tuning) for details.*
- models.generate_answer(model, contents, answer_style, inline_passages, semantic_retriever, safety_settings, temperature): *Generates a grounded answer from the model given an input `GenerateAnswerRequest`.*
- models.stream_generate_content(model, contents, system_instruction, tools, tool_config, safety_settings, generation_config, cached_content, service_tier, store): *Generates a [streamed response](https://ai.google.dev/gemini-api/docs/text-generation?lang=python#generate-a-text-stream) from the model given an input `GenerateContentRequest`.*
- models.embed_content(model, content, task_type, title, output_dimensionality): *Generates a text embedding vector from the input `Content` using the specified [Gemini Embedding model](https://ai.google.dev/gemini-api/docs/models/gemini#text-embedding).*
- models.batch_embed_contents(model, requests): *Generates multiple embedding vectors from the input `Content` which consists of a batch of strings represented as `EmbedContentRequest` objects.*
- models.count_tokens(model, contents, generate_content_request): *Runs a model's tokenizer on input `Content` and returns the token count. Refer to the [tokens guide](https://ai.google.dev/gemini-api/docs/tokens) to learn more about tokens.*
- models.batch_generate_content(model, batch): *Enqueues a batch of `GenerateContent` requests for batch processing.*
- models.async_batch_embed_content(model, batch): *Enqueues a batch of `EmbedContent` requests for batch processing. We have a `BatchEmbedContents` handler in `GenerativeService`, but it was synchronized. So we name this one to be `Async` to avoid confusion.*
- models.generate_message(model, prompt, temperature, candidate_count, top_p, top_k): *Generates a response from the model given an input `MessagePrompt`.*
- models.count_message_tokens(model, prompt): *Runs a model's tokenizer on a string and returns the token count.*
- models.get(name): *Gets information about a specific `Model` such as its version number, token limits, [parameters](https://ai.google.dev/gemini-api/docs/models/generative-models#model-parameters) and other metadata. Refer to the [Gemini models guide](https://ai.google.dev/gemini-api/docs/models/gemini) for detailed model information.*
- models.list(page_size, page_token): *Lists the [`Model`s](https://ai.google.dev/gemini-api/docs/models/gemini) available through the Gemini API.*
- models.predict(model, instances, parameters): *Performs a prediction request.*
- models.predict_long_running(model, instances, parameters): *Same as Predict but returns an LRO.*
- models.generate_text(model, prompt, temperature, candidate_count, max_output_tokens, top_p, top_k, safety_settings, stop_sequences): *Generates a response from the model given an input message.*
- models.embed_text(model, text): *Generates an embedding from the model given an input message.*
- models.batch_embed_text(model, texts, requests): *Generates multiple embeddings from the model given input text in a synchronous call.*
- models.count_text_tokens(model, prompt): *Runs a model's tokenizer on a text and returns the token count.*

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

### Delta

Canonical streaming event fragment

In [ ]:
#| export
@dataclass
class Delta:
    "Normalized streaming delta event."
    text: str = ""
    thinking: str = ""
    refusal: str = ""
    tool_calls: list[ToolCall] = field(default_factory=list)
    citations: list = field(default_factory=list)
    server_tool_result: dict = None
    finish_reason: str = None
    usage: Usage = None
    raw: dict = field(default_factory=dict)

In [ ]:
#| export
async def norm_and_yield(resp, norm_func):
    async for ev in resp:
        if (d := norm_func(ev)) is not None: yield d

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o):
        if not isinstance(o, Completion): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

`ToolCall` normalizes a tool call with `id`, `name`, and `arguments`, anything extra goes into `extra`

`tool_use` parts are built from tool calls, and store `ToolCall` fields in `data`, which will be converted into correct message type based on provider, and allow users to pass `Msg` directly to the chat history

### PartAccum

Collator for tool calls and other text parts

In [ ]:
#| export
@dataclass
class PartAccum:
    parts: dict[Part|ToolCall] = field(default_factory=dict)
    tool_calls: list[ToolCall] = field(default_factory=list)
    
    def append(self, typ, index, txt='', citations=None, **tc_kwargs):
        'Create and accumulate same type sequential parts'
        if index not in self.parts: 
            if typ==PartType.tool_use: self.parts[index] = ToolCall(**tc_kwargs)
            else:                      self.parts[index] = Part(type=typ, text=txt, data=dict(citations=citations or []))
        else:
            if typ==PartType.tool_use:
                    new_args = tc_kwargs.get('arguments', '')
                    cur_args = self.parts[index].arguments
                    if isinstance(new_args, str) and isinstance(cur_args, str): self.parts[index].arguments += new_args
                    elif isinstance(new_args, str) and isinstance(cur_args, dict): self.parts[index].arguments = new_args
                    else: self.parts[index].arguments = new_args
            else: 
                self.parts[index].text += txt
                # anthropic citations have matching idx
                self.parts[index].data['citations'].extend(citations or [])
    
    def finalize(self, api_name=None):
        for idx,tc in self.parts.items():
            if isinstance(tc, ToolCall):
                if isinstance(tc.arguments, str): tc.arguments = json.loads(tc.arguments) if tc.arguments else {}
                self.tool_calls.append(tc)
                self.parts[idx] = Part(type=PartType.tool_use, data={**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server})
        
        # No need to preserve citations in Part.data, we just emit in `_acollect_stream`
        merged = []
        for p in self.parts.values():
            if merged and merged[-1].type == p.type and p.type in (PartType.text, PartType.thinking): merged[-1].text += p.text
            else: merged.append(p)
        
        self.parts = merged

### _accolect_stream

A generic `_acollect_stream` function that yields thinking and text (if needed we can yield tool calls to), and collates parts while keeping the order. A custom `index_fn` is used to resolve the index that the current `Delta` event belongs to.

In [ ]:
#| export
async def _acollect_stream(it, index_fn, model=None, api_name=None, vendor_name=None):
    "Collect a Delta stream, yielding incremental chunks and a final Completion."
    part_accum = PartAccum()
    deltas = []
    typ, last_typ, last_idx, last_idx = None, None, None, None
    async for d in it:
        if d.text:     
            yield {'text': d.text}
            typ = PartType.text
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.text)
        if d.thinking: 
            yield {'thinking': d.thinking}
            typ = PartType.thinking
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.thinking)
        if d.citations:
            yield {'citations': d.citations} # anthropic
            typ = PartType.text
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, citations=d.citations)
        for tc in d.tool_calls:
            args = tc.arguments.get('_delta', '') if '_delta' in tc.arguments else tc.arguments
            tc_kwargs = dict(id=tc.id, name=tc.name, arguments=args, server=tc.server, extra=tc.extra)
            typ = PartType.tool_use
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, **tc_kwargs)
        if d.server_tool_result: 
            typ = PartType.server_tool_result
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.parts[idx] = Part(type=typ, data=d.server_tool_result)
        if d.refusal:
            typ = PartType.refusal
            idx,last_idx = index_fn(d, typ, last_typ, last_idx)
            part_accum.append(typ, idx, d.refusal)        
        if d.finish_reason: fin = d.finish_reason
        if d.usage: usg = d.usage
        last_typ = typ
        deltas.append(d)
    part_accum.finalize(api_name)
    fin = canon_finish(fin, api_name, part_accum.tool_calls) # need to canon one more time with accum'd tool calls
    # tool calls and non-anthropic citations are yielded at the end
    yield Completion(d.raw.get('model', model),
            message=Msg(role="assistant", content=part_accum.parts),
            finish_reason=fin,
            usage=usg,
            tool_calls=part_accum.tool_calls,
            api_name=api_name,
            vendor_name=vendor_name,
            raw={'deltas':deltas})

### OpenAI Responses Events

~~OpenAI responses api returns collated responses in the last stream delta, so we can build the final `Completion` object using the `normalize_openai_response` function without needing anything else:~~

This doesn't hold for Codex so we've rewritten it like below and use the existing `PartAccum` and `_acollect_stream`

In [ ]:
#| export
def normalize_openai_response_event(ev, **kwargs):
    "Normalize OpenAI Responses API stream event into Delta."
    typ = ev.get("type")
    if typ == "response.output_text.delta":            return Delta(text=ev.get("delta"), raw=ev, **kwargs)
    if typ == "response.reasoning_text.delta":         return Delta(thinking=ev.get("delta",""), raw=ev, **kwargs)
    if typ == "response.reasoning_summary_text.delta": return Delta(thinking=ev.get("delta",""), raw=ev, **kwargs)
    if typ == "response.output_text.annotation.added":
        return Delta(citations=[ev.get("annotation")], raw=ev, **kwargs)
    if typ == "response.output_item.done":
        item = ev.get("item", {})
        itype = item.get("type", "")
        if itype == "function_call" or itype.endswith("_call"):
            tc = openai_responses_tool_call(item)
            return Delta(tool_calls=[tc], raw=ev, **kwargs)
    if typ in ("response.completed", "response.incomplete"):
        resp = ev.get("response", {})
        fin = canon_finish(resp.get("status"), ApiName.openai)
        return Delta(finish_reason=fin, usage=usage_from_openai(resp), raw=ev, **kwargs)
    if typ == "error": raise api_error_from_event(ev)

In [ ]:
#| export
def openai_responses_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    idx = nested_idx(d, 'raw', 'output_index')
    return idx, idx

In [ ]:
#| export
_acollect_stream_openai_responses = partial(_acollect_stream, index_fn=openai_responses_index_fn, api_name=ApiName.openai)

#### Text

In [ ]:
mn,inp = 'gpt-4o-mini','Hi!'
resp = await oai_cli.responses.create_response(model=mn,input=inp)
comp = normalize_openai_response(resp, mn)

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, stream=True)
# async for o in  acollect_stream(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)
async for o in  _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

Hello

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='Hello! How can I assist you today?', data={'type': 'output_text', 'logprobs': [], 'text': 'Hello! How can I assist you today?', 'citations': []})], data=None)

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='Hello! How can I assist you today?', data={'citations': []})], data=None)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.usage, o.usage)
test_eq(comp.tool_calls, o.tool_calls)

#### Thinking

In [ ]:
mn,inp = 'o4-mini','What is 31231231 * 12312?'
resp = await oai_cli.responses.create_response(model=mn,input=inp,reasoning={"effort": "low", "summary": "auto"})
comp = normalize_openai_response(resp, mn)

In [ ]:
resp = await oai_cli.responses.create_response(model=mn,input=inp,reasoning={"effort": "low", "summary": "auto"},stream=True)
# async for o in acollect_stream(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)
async for o in _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

🧠

🧠

🧠

🧠

🧠

...

,

916

,

072

.

In [ ]:
for p in o.message.content: print(p,'\n')

Part(type=<PartType.thinking: 'thinking'>, text='**Calculating multiplication**\n\nThe user’s request is a multiplication problem: 31,231,231 × 12,312. I’m going to compute this using long multiplication. A better approach is to break down 12,312 into 12,000 + 312. \n\nSo, multiplying step-by-step: \n\n- 31,231,231 * 12,000 = 374,774,772,000. \n- 31,231,231 * 312 = 9,744,144,072.\n  \nAdding both parts, I find the total to be 384,518,916,072. I’ll double-check the math just to be sure, and everything matches up!**Verifying multiplication**\n\nI did the calculations for 31.231231 * 12 and got 374.774772. Then I checked .312 * 31.231231, which is around 9.744. Adding those together gives me approximately 384.518772, which matches my previous result. Everything seems correct! \n\nSo, I’ll report the final product: 384,518,916,072, since that’s what the user is looking for. This should give them the answer they need!', data={'citations': []}) 

Part(type=<PartType.text: 'text'>, text='The 

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
mn,inp = 'gpt-4o-mini','What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'
tools=[{"type": "function", **sch['function']}]
resp = await oai_cli.responses.create_response(model=mn,input=inp,tools=tools)
comp = normalize_openai_response(resp, mn)

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
# async for o in  acollect_stream(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)
async for o in _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

In [ ]:
comp.tool_calls

[ToolCall(id='fc_07079e5b966f6d8b0069eb7881b63081a081d4528e9b501c06', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_WVJMupWoABKhiYqAFUoj2wPX'}),
 ToolCall(id='fc_07079e5b966f6d8b0069eb7881b63c81a0aac92589ffa58a58', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_gwfIaqLA4Vkg24uaWHg0zFsk'})]

In [ ]:
o.tool_calls

[ToolCall(id='fc_08f4fa7c0518eb7b0069eb788384fc8191bc6797f60434c683', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_JJRoiKaTHWskP1RCSzazaqWr'}),
 ToolCall(id='fc_08f4fa7c0518eb7b0069eb7883851081919dfde479b4a560ea', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function_call', 'status': 'completed', 'call_id': 'call_Fn8hwJ9noxaiMvIGBMH485lP'})]

In [ ]:
comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_07079e5b966f6d8b0069eb7881b63081a081d4528e9b501c06', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_WVJMupWoABKhiYqAFUoj2wPX'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_07079e5b966f6d8b0069eb7881b63c81a0aac92589ffa58a58', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_gwfIaqLA4Vkg24uaWHg0zFsk'})]

In [ ]:
o.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_08f4fa7c0518eb7b0069eb788384fc8191bc6797f60434c683', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_JJRoiKaTHWskP1RCSzazaqWr'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_08f4fa7c0518eb7b0069eb7883851081919dfde479b4a560ea', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_Fn8hwJ9noxaiMvIGBMH485lP'})]

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(set(comp.message.content[0].data.keys()), set(o.message.content[0].data.keys()))
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.usage, o.usage)

#### Web search

In [ ]:
mn,inp = 'gpt-4o-mini','What is the weather in Istanbul today?'
tools=[{"type": "web_search_preview"}]
resp = await oai_cli.responses.create_response(model=mn,input=inp,tools=tools)
comp = normalize_openai_response(resp, mn)

OpenAI responses text parts already include citations inline, but we still include it in `Part.data`:

In [ ]:
cites = comp.message.content[-1].data['citations']; cites

[{'type': 'url_citation',
  'end_index': 933,
  'start_index': 830,
  'title': 'April weather - Spring 2026 - Istanbul, Turkey',
  'url': 'https://www.weather-atlas.com/en/turkey/istanbul-weather-april?utm_source=openai'}]

In [ ]:
Markdown(comp.message.content[-1].text)

<div class="prose">

As of 5:04 PM local time in Istanbul on April 24, 2026, the weather is sunny with a temperature of 64°F (18°C).

## Weather for Hastane, Hadımköy-İstanbul Caddesi 34, 34555 Arnavutköy/İstanbul, Türkiye:
Current Conditions: Sunny, 64°F (18°C)

Hourly Forecast:
* 6:00 PM: 62°F (17°C), Mostly sunny
* 7:00 PM: 59°F (15°C), Partly sunny
* 8:00 PM: 57°F (14°C), Mostly clear
* 9:00 PM: 54°F (12°C), Clear
* 10:00 PM: 52°F (11°C), Clear
* 11:00 PM: 49°F (10°C), Clear
* 12:00 AM: 49°F (9°C), Clear
* 1:00 AM: 48°F (9°C), Clear
* 2:00 AM: 48°F (9°C), Clear
* 3:00 AM: 47°F (8°C), Clear
* 4:00 AM: 47°F (8°C), Mostly clear
* 5:00 AM: 45°F (7°C), Partly cloudy


Historically, April in Istanbul experiences average high temperatures around 59.5°F (15.3°C) and lows near 49.8°F (9.9°C), with approximately 29mm of rainfall over 12.6 days. ([weather-atlas.com](https://www.weather-atlas.com/en/turkey/istanbul-weather-april?utm_source=openai))

Today, the weather is slightly warmer than average, with clear skies and minimal chance of precipitation. 

</div>

In [ ]:
comp.message.content[-1].text[cites[0]['start_index']:]

'([weather-atlas.com](https://www.weather-atlas.com/en/turkey/istanbul-weather-april?utm_source=openai))\n\nToday, the weather is slightly warmer than average, with clear skies and minimal chance of precipitation. '

Same with streaming:

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
async for o in _acollect_stream_openai_responses(norm_and_yield(resp, normalize_openai_response_event)): print_stream(o)

As

 of

5

:

04

...

 minimal

 chance

 of

 precipitation

.

In [ ]:
cites = o.message.content[-1].data['citations']; cites

[{'type': 'url_citation',
  'end_index': 933,
  'start_index': 830,
  'title': 'April weather - Spring 2026 - Istanbul, Turkey',
  'url': 'https://www.weather-atlas.com/en/turkey/istanbul-weather-april?utm_source=openai'}]

In [ ]:
o.message.content[-1].text[cites[0]['start_index']:]

'([weather-atlas.com](https://www.weather-atlas.com/en/turkey/istanbul-weather-april?utm_source=openai))\n\nToday, the weather is slightly warmer than average, with clear skies and minimal chance of precipitation. '

In [ ]:
comp.tool_calls

[ToolCall(id='ws_014a600da66b942c0069eb7884e6fc8191ad64ffed70b7f64b', name='web_search', arguments={'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}})]

In [ ]:
o.tool_calls

[ToolCall(id='ws_0b20526b4659f8bf0069eb7888d8e8819e92623ea7a30acf69', name='web_search', arguments={'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}, server=True, extra={'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}})]

In [ ]:
comp.finish_reason, o.finish_reason

('stop', 'stop')

In [ ]:
comp.usage, o.usage

(Usage(prompt_tokens=309, completion_tokens=409, total_tokens=718, raw={'input_tokens': 309, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 409, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 718, 'server_tool_use': {'web_search_call': 1}}),
 Usage(prompt_tokens=309, completion_tokens=409, total_tokens=718, raw={'input_tokens': 309, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 409, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 718, 'server_tool_use': {'web_search_call': 1}}))

### OpenAI Chat Events

In [ ]:
#| export
def normalize_openai_chat_delta(ev, **kwargs):
    "Normalize a chat completion stream event."
    # usage always arrives as a single final event with choices: []
    if not (choices := ev.get("choices", [])): return Delta(usage=usage_from_openai(ev), raw=ev, **kwargs)
    # finish_reason arrives in its own dedicated chunk (empty delta, non-null finish_reason)
    if fin := nested_idx(ev, 'choices', 0, 'finish_reason'): return Delta(finish_reason=fin, raw=ev, **kwargs)
    # repurposed the common function
    tcs = openai_chat_tool_calls(ev, delta=True)
    dlt = nested_idx(choices, 0, 'delta')
    if ev.get("error"): raise api_error_from_event(ev)
    return Delta(text=dlt.get('content'), thinking=dlt.get('reasoning_content'), refusal=dlt.get('refusal'), tool_calls=tcs, raw=ev, **kwargs)

In [ ]:
#| export
def openai_chat_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    if d.tool_calls: 
        tc_idx = nested_idx(d.tool_calls, 0, 'extra', 'index')
        return f"tool_{tc_idx}", last_idx
    if not (last_typ or last_idx): return 0,0
    if typ == last_typ: return last_idx, last_idx
    return last_idx + 1, last_idx + 1

In [ ]:
#| export
_acollect_stream_openai_chat = partial(_acollect_stream, index_fn=openai_chat_index_fn, api_name=ApiName.openai_chat)

#### Text

In [ ]:
mn,msgs = 'gpt-4o-mini',[{"role": "user", "content": "Say hi"}]
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs)
comp = normalize_openai_chat_completion(resp, mn)

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs,stream=True,stream_options={"include_usage": True})
async for o in _acollect_stream_openai_chat(norm_and_yield(resp, normalize_openai_chat_delta)): print_stream(o)

Hi

 there

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type='text', text='Hi there! How can I assist you today?', data={'citations': []})], data=None)

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text='Hi there! How can I assist you today?', data={'citations': []})], data=None)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, raw={'prompt_tokens': 9, 'completion_tokens': 10, 'total_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}),
 Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, raw={'prompt_tokens': 9, 'completion_tokens': 10, 'total_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}))

#### Thinking

OpenAI's Chat Completions API doesn't expose reasoning content — `reasoning_tokens` appear in usage but no `reasoning_content` field is returned. We use Kimi (`kimi-k2.5`) via Moonshot's OpenAI-compatible API to test thinking parts in chat completions streaming.

**NB**: Kimi's thinking is controlled via a `thinking` body param (not `reasoning_effort`). Use `_body={"thinking": {"type": "disabled"}}` to disable it, or `_body={"thinking": {"type": "enabled"}}` to enable. Since `thinking` isn't in the OpenAI spec, `fastspec` requires the `_body` escape hatch.

Kimi's thinking is binary — enabled (default) or disabled. There's no `reasoning_effort` level (low/medium/high) or `budget_tokens` like OpenAI/Anthropic.

**TODO**: Expose `_body`/`native` escape hatches to users in the high-level API so provider-specific params (like Kimi's `thinking`) can be passed through without needing to drop down to raw `fastspec` calls.

In [ ]:
mn,msgs = 'kimi-k2.5',[{"role": "user", "content": 'What is 31231231 * 12312?'}]
resp = await kimi_cli.chat.create_chat_completion(model=mn,messages=msgs,reasoning_effort='low')
comp = normalize_openai_chat_completion(resp, mn)

In [ ]:
resp = await kimi_cli.chat.create_chat_completion(model=mn,messages=msgs,stream=True,stream_options={"include_usage": True})
async for o in _acollect_stream_openai_chat(norm_and_yield(resp, normalize_openai_chat_delta)): print_stream(o)

🧠

🧠

🧠

🧠

🧠

...

518

,

916

,

072

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
mn = 'gpt-4o-mini'
msgs = [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs, tools=[sch])
comp = normalize_openai_chat_completion(resp, mn)

In [ ]:
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs, tools=[sch], stream=True, stream_options={"include_usage": True})
async for o in _acollect_stream_openai_chat(norm_and_yield(resp, normalize_openai_chat_delta)): print_stream(o)

In [ ]:
comp.tool_calls

[ToolCall(id='call_yY8RSMwXW6lvsdUiFq1rqWTr', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'type': 'function'}),
 ToolCall(id='call_lyxh05ANtgEcIA9gcVGEkavV', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'type': 'function'})]

In [ ]:
o.tool_calls

[ToolCall(id='call_VGNQoGfoMeHB7jEeKNhvRkIX', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'index': 0, 'type': 'function'}),
 ToolCall(id='call_pwlG3QyTKHCx160PTmem5FKU', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'index': 1, 'type': 'function'})]

In [ ]:
comp.message.content

[Part(type='tool_use', text=None, data={'id': 'call_yY8RSMwXW6lvsdUiFq1rqWTr', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function'}),
 Part(type='tool_use', text=None, data={'id': 'call_lyxh05ANtgEcIA9gcVGEkavV', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function'})]

In [ ]:
o.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_VGNQoGfoMeHB7jEeKNhvRkIX', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'index': 0, 'type': 'function'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_pwlG3QyTKHCx160PTmem5FKU', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'index': 1, 'type': 'function'})]

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}),
 Usage(prompt_tokens=66, completion_tokens=52, total_tokens=118, raw={'prompt_tokens': 66, 'completion_tokens': 52, 'total_tokens': 118, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}))

### Anthropic Messages Events

In [ ]:
#| export
def normalize_anthropic_event(ev, **kwargs):
    typ = ev.get("type")
    text, thinking, tcs, citations = None, None, [], None
    if typ == "content_block_start":
        cb = ev.get("content_block", {})
        if cb.get("type", "").endswith("_tool_result"): return Delta(server_tool_result=cb, raw=ev, **kwargs)
        if tc := anthropic_tool_call(cb): tcs = [tc]
    elif typ == "content_block_delta":
        d = ev.get("delta", {})
        dtyp = d.get("type")
        if   dtyp == "text_delta": text = d.get("text")
        elif dtyp == "thinking_delta": thinking = d.get("thinking")
        elif dtyp == "input_json_delta":
            tcs = [ToolCall(id=str(ev.get("index", "")), name="", arguments={"_delta": d.get("partial_json", '')})]
        elif dtyp == "citations_delta": 
            citations = listify(d.get('citation',[]))
    elif typ == "message_delta":
        fin = canon_finish(nested_idx(ev, 'delta', 'stop_reason'), 'anthropic')
        return Delta(finish_reason=fin, usage=usage_from_anthropic(ev), raw=ev, **kwargs)
    elif typ == "error": raise api_error_from_event(ev)
    return Delta(text=text, thinking=thinking, tool_calls=tcs, citations=citations, raw=ev, **kwargs)

In [ ]:
#| export
def anthropic_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    return nested_idx(d, 'raw', 'index'), None

In [ ]:
#| export
_acollect_stream_anthropic = partial(_acollect_stream, index_fn=anthropic_index_fn, api_name=ApiName.anthropic)

#### Thinking

In [ ]:
mn,msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "Say hi"}]
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, max_tokens=8000,
                                            thinking=think, headers=hdrs)
comp = normalize_anthropic_message(resp, mn)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn,messages=msgs,max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for o in _acollect_stream_anthropic(norm_and_yield(resp, normalize_anthropic_event)): print_stream(o)

🧠

🧠

Hi there! How are you doing today?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type=<PartType.thinking: 'thinking'>, text='The user is asking me to say hi, which is a simple greeting. I should respond in a friendly and warm manner.', data={'type': 'thinking', 'thinking': 'The user is asking me to say hi, which is a simple greeting. I should respond in a friendly and warm manner.', 'signature': 'ErACCmIIDRgCKkAfKDm83L6l1GfsuCsA/mF3NWz9U5HRK/B8T3KMFIbi4GXXfDJdbK2ZL/aynZ1peLcWF6X5ujZ5JuUdEUtdzDgRMhhjbGF1ZGUtc29ubmV0LTQtMjAyNTA1MTQ4ABIMT03E+GG2HahiEeXrGgyEng61OLMUUqkM8m8iMCuLQr9gdVkZJhusrc3l1TNH0NOhIljxMBD5LSX3nKOx2Y4jd8+oMFvkOCeutJSrZSp8PaOPwPqlAZOo/sRcIPSKLZllVuOEoQCTr6+V3WxUjSU1K1VSk9xh48zOwBIHlv/Eo2ygYMVdDxjhFuKtJFMhNaZeYm90VxM+I3jn39S7Yj4MtVz/iwafdY1kM2P65LvKvZ85IYdzr7ijcAPeJQaR+Tk2bXopKBBkKQ+lexgB'}), Part(type=<PartType.text: 'text'>, text='Hi there! How are you doing today?', data={'type': 'text', 'text': 'Hi there! How are you doing today?'})], data=None)

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.thinking: 'thinking'>, text='The user is asking me to say hi. This is a simple, friendly greeting request. I should respond in a warm and friendly manner.', data={'citations': []}), Part(type=<PartType.text: 'text'>, text='Hi there! How are you doing today?', data={'citations': []})], data=None)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
mn,msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools = [{"name": "simple_add", "description": "add numbers", "input_schema": sch['function']['parameters']}]
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs)
comp = normalize_anthropic_message(resp, mn)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs, stream=True)
async for o in _acollect_stream_anthropic(norm_and_yield(resp, normalize_anthropic_event)): print_stream(o)

🧠

🧠

🧠

🧠

🧠

I'll calculate both additions for you using the simple_add tool in parallel.

In [ ]:
comp.tool_calls

[ToolCall(id='toolu_011CeeKRhcwGnhTVJjXDUVf1', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
 ToolCall(id='toolu_016TzK4mfYWcZz7qsZLwAACd', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})]

In [ ]:
o.tool_calls

[ToolCall(id='toolu_01X5TSjVQQ8H5unY3KjLamin', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
 ToolCall(id='toolu_01YStL1e5tK6aU4o1i5Pz5vE', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})]

In [ ]:
comp.message.content

[Part(type=<PartType.thinking: 'thinking'>, text="The user is asking me to calculate two addition problems:\n1. 3 + 5\n2. 10 + 20\n\nThey specifically want me to use the simple_add tool and do this in parallel. I have access to the simple_add function which takes parameters 'a' and 'b' (both integers). I can make both function calls simultaneously.\n\nFor the first calculation: a=3, b=5\nFor the second calculation: a=10, b=20\n\nI have all the required parameters for both calls, so I can proceed.", data={'type': 'thinking', 'thinking': "The user is asking me to calculate two addition problems:\n1. 3 + 5\n2. 10 + 20\n\nThey specifically want me to use the simple_add tool and do this in parallel. I have access to the simple_add function which takes parameters 'a' and 'b' (both integers). I can make both function calls simultaneously.\n\nFor the first calculation: a=3, b=5\nFor the second calculation: a=10, b=20\n\nI have all the required parameters for both calls, so I can proceed.", 'si

In [ ]:
o.message.content

[Part(type=<PartType.thinking: 'thinking'>, text='The user is asking me to calculate two additions:\n1. 3 + 5\n2. 10 + 20\n\nThey want me to use the simple_add tool in parallel, which means I should make both function calls at the same time.\n\nLooking at the simple_add function, it takes two required parameters:\n- a: integer\n- b: integer\n\nFor the first calculation (3 + 5):\n- a = 3\n- b = 5\n\nFor the second calculation (10 + 20):\n- a = 10\n- b = 20\n\nI have all the required parameters, so I can proceed with both function calls in parallel.', data={'citations': []}),
 Part(type=<PartType.text: 'text'>, text="I'll calculate both additions for you using the simple_add tool in parallel.", data={'citations': []}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_01X5TSjVQQ8H5unY3KjLamin', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'caller': {'type': 'direct'}}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'to

In [ ]:
test_eq(comp.tool_calls[0].server, False)
for tc in comp.tool_calls: tc.id = '123'
for tc in o.tool_calls:    tc.id = '123'
test_eq(comp.tool_calls, o.tool_calls)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=447, completion_tokens=270, total_tokens=717, raw={'input_tokens': 447, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 270, 'service_tier': 'standard', 'inference_geo': 'not_available'}),
 Usage(prompt_tokens=447, completion_tokens=295, total_tokens=742, raw={'input_tokens': 447, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 295}))

#### Web search

In [ ]:
def add_ant_link(citations):
    links = []
    for c in citations:
        if 'title' not in c: return ''
        title = c['title'].replace('"', '\\"')
        links.append(f'[*]({c["url"]} "{title}")')
    return ' '.join(links)

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o):
        if not isinstance(o, Completion): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            if citations:= o.get('citations'): print(add_ant_link(citations),end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

In [ ]:
mn,msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "Use web search to get the weather in Istanbul?"}]
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools=[{"type": "web_search_20250305", "name": "web_search"}]
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs)
comp = normalize_anthropic_message(resp, mn)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs, stream=True)
async for o in _acollect_stream_anthropic(norm_and_yield(resp, normalize_anthropic_event)): print_stream(o)

🧠

🧠

🧠

Based

 on the search results, here's the current weather information for Istanbul:

**Current Weather in Istanbul:**

...

[*](https://www.accuweather.com/en/tr/istanbul/318251/weather-forecast/318251 "Istanbul, Istanbul, Türkiye Weather Forecast | AccuWeather")

Air quality is listed

 as "Unhealthy"



The weather appears to be pleasant with clear to mostly clear skies and

 mild temperatures typical for late April in Istanbul.

No collation done in the non-streaming path, response parts are returned as is:

In [ ]:
len(comp.message.content)

22

~~Citations are attached to each `Part.data`:~~ citations are streamed in `_acollect_stream`

In [ ]:
text_cites = L(comp.message.content).map(lambda p: (p.text, p.data.get('citations'))).filter(lambda o:o[1]); text_cites[:3]

[('Istanbul is currently mostly clear with a temperature of 46°F', [{'type': 'web_search_result_location', 'cited_text': 'LEARN MORE ... Istanbul, Istanbul is currently Mostly clear with a temperature of 46°. ', 'url': 'https://www.accuweather.com/en/tr/istanbul/318251/current-weather/318251', 'title': 'Istanbul, Istanbul, Türkiye Current Weather | AccuWeather', 'encrypted_index': 'Eo8BCioIDxgCIiQ4ODk4YTFkYy0yMTNkLTRhNmYtOTljYi03ZTBlNTUzZDc0NWISDDUVjtLGXsb6ETbn3RoMK/JgBZ4+f+H2ygUXIjD6asPFWj93uVXz3xA0qWCFd/ZJ7bd/R1us1F3kOw2YS/RHRpSfzQBIUnnWEPSDxqgqEwpXOIG8ENl6jSmg37WQW04ybFQYBA=='}]), ('The humidity is at 74% with winds from the SSW at 5 mph', [{'type': 'web_search_result_location', 'cited_text': 'LEARN MORE · RealFeel® · 45° · Wind · SSW 5 mph · Wind Gusts · 5 mph · Humidity · 74% Indoor Humidity · 34% (Slightly Dry) Dew Point · 39° F · Pressur...', 'url': 'https://www.accuweather.com/en/tr/istanbul/318251/current-weather/318251', 'title': 'Istanbul, Istanbul, Türkiye Current Weather |

In [ ]:
t, c = text_cites[0][0], text_cites[0][1][0]; c

{'type': 'web_search_result_location',
 'cited_text': 'LEARN MORE ... Istanbul, Istanbul is currently Mostly clear with a temperature of 46°. ',
 'url': 'https://www.accuweather.com/en/tr/istanbul/318251/current-weather/318251',
 'title': 'Istanbul, Istanbul, Türkiye Current Weather | AccuWeather',
 'encrypted_index': 'Eo8BCioIDxgCIiQ4ODk4YTFkYy0yMTNkLTRhNmYtOTljYi03ZTBlNTUzZDc0NWISDDUVjtLGXsb6ETbn3RoMK/JgBZ4+f+H2ygUXIjD6asPFWj93uVXz3xA0qWCFd/ZJ7bd/R1us1F3kOw2YS/RHRpSfzQBIUnnWEPSDxqgqEwpXOIG8ENl6jSmg37WQW04ybFQYBA=='}

In [ ]:
title = c['title'].replace('"', '\\"')
Markdown(f'[*]({c["url"]} "{title}") {t}')

<div class="prose">

[*](https://www.accuweather.com/en/tr/istanbul/318251/current-weather/318251 "Istanbul, Istanbul, Türkiye Current Weather | AccuWeather") Istanbul is currently mostly clear with a temperature of 46°F

</div>

Same with streaming:

In [ ]:
L(comp.message.content).attrgot('type'), L(o.message.content).attrgot('type')

([<PartType.thinking: 'thinking'>, <PartType.tool_use: 'tool_use'>, <PartType.server_tool_result: 'server_tool_result'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>, <PartType.text: 'text'>],
 [<PartType.thinking: 'thinking'>, <PartType.tool_use: 'tool_use'>, <PartType.server_tool_result: 'server_tool_result'>, <PartType.text: 'text'>])

In [ ]:
# text_cites = L(o.message.content).map(lambda p: (p.text, p.data.get('citations'))).filter(lambda o:o[1]); text_cites[:3]

In [ ]:
# t, c = text_cites[0][0], text_cites[0][1][0]; c

In [ ]:
title = c['title'].replace('"', '\\"')
Markdown(f'[*]({c["url"]} "{title}") {t}')

<div class="prose">

[*](https://www.accuweather.com/en/tr/istanbul/318251/current-weather/318251 "Istanbul, Istanbul, Türkiye Current Weather | AccuWeather") Istanbul is currently mostly clear with a temperature of 46°F

</div>

In [ ]:
test_eq(comp.tool_calls[0].server, True)
for tc in comp.tool_calls: tc.id = '123'
for tc in o.tool_calls:    tc.id = '123'
test_eq(comp.tool_calls, o.tool_calls)

In [ ]:
test_eq('server_tool_result' in L(comp.message.content).attrgot('type'), True)
test_eq('server_tool_result' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=11098, completion_tokens=438, total_tokens=11536, raw={'input_tokens': 11098, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 438, 'service_tier': 'standard', 'inference_geo': 'not_available', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}),
 Usage(prompt_tokens=11093, completion_tokens=475, total_tokens=11568, raw={'input_tokens': 11093, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 475, 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}))

### Gemini Generate Events

In [ ]:
#| export
def normalize_gemini_event(ev, **kwargs):
    "Normalize Gemini stream event into Delta."
    cand = nested_idx(ev, 'candidates', 0) or {}
    finish_reason = canon_finish(cand.get("finishReason"), 'gemini')
    parts = nested_idx(cand, 'content', 'parts') or []
    thinking = "".join(p.get("text","") for p in parts if p.get("thought") and "text" in p)
    txt = "".join(p.get("text","") for p in parts if not p.get("thought") and "text" in p)
    tcs = gemini_tool_calls(ev)
    if ev.get("error"): raise api_error_from_event(ev)
    return Delta(text=txt, thinking=thinking, tool_calls=tcs, citations=listify(cand.get('groundingMetadata', [])), finish_reason=finish_reason, usage=usage_from_gemini(ev), raw=ev, **kwargs)

In [ ]:
#| export
def gemini_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    if not (last_typ or last_idx): return 0,0
    return last_idx + 1, last_idx + 1

In [ ]:
#| export
_acollect_stream_gemini = partial(_acollect_stream, index_fn=gemini_index_fn, api_name=ApiName.gemini)

#### Text

In [ ]:
mn,inp = 'models/gemini-3-flash-preview',[{"role": "user", "parts": [{"text": "Hi how are you?"}]}]
resp = await gem_cli.models.generate_content(model=mn,contents=inp)
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

I'm doing great, thank you for asking! How are you doing today? Is

 there anything I can help you with?

In [ ]:
comp.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text="I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?", data={'text': "I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?", 'thoughtSignature': 'Eu0GCuoGAQw51scu9RoJhmMFR7Wzp00X0VvB3bFTS5nFz20FSKYZKfcT88tZUuRjiikofz3kg8Y/JbhrkF0uOU427XK+M5G60b5E2gKSg6JghlOo03BL3vH4u7gC9pwTnfc2A1dOmNGhlhW4b9Sf8FpvmjG/yq9kW0EeTqOnF8Y/8fpVinKJZMRYLrZ5KINGDC128fZYexkBjgBgqpDW4H/NMuLn5KjmRQqsno8IPA1z9gQvVXsFf+hPRvk9qVkpvuw81I0qq/P4/V6pEURiwxcYs5y8PWxTuQdq/RFNKVHObFf2CQyS43zkvzLj2mrv9IM1efBEXykqxa6AT419Zc0vYHU15nsJSzePivMOEQbZnew1RKp/HwXeYJY6DlNqtlIzf+BOhfMgI2/HYtWOWtMo3qMMjDE2LM4IoazWmbIRyjDJN+yU5QkulztcwDN4f0JFVKJHns9CRq1RAY7mBBKaeIuMp5UNmepLd3XH4NTIiKwz1pNLQQTavBMnB6uqtWyX1y2hdSPWKqifoAqzggfNZ/YFfgglSBOQGq8qz813+P0kmzWtQQ9zzt1nZX9Pm+02sFa+kPdbtjGn29lRWrOsFkTXLH3O0DYhOhttxe2cM2YT86SXmQywuTMKQecv9+W/TzPiNaNN8lYRx/RXbRWSO/GJKC4UR2V1yjDDiT

In [ ]:
o.message

Msg(role='assistant', content=[Part(type=<PartType.text: 'text'>, text="I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?", data={'citations': []})], data=None)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=6, completion_tokens=26, total_tokens=274, raw={'promptTokenCount': 6, 'candidatesTokenCount': 26, 'totalTokenCount': 274, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 242}),
 Usage(prompt_tokens=6, completion_tokens=26, total_tokens=212, raw={'promptTokenCount': 6, 'candidatesTokenCount': 26, 'totalTokenCount': 212, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 180}))

#### Thinking

- **Thinking always happens internally** for Gemini 3/2.5 models — token cost visible in `thoughtsTokenCount` in usage metadata
- **Thought text in response** only returned when `include_thoughts: True` is explicitly set in `generation_config.thinking_config` — defaults to `false`
- **`thoughtSignature`** is always returned regardless of `includeThoughts` — required for multi-turn context continuity. Missing signatures in function calling result in a 400 error; for text/chat, omitting them degrades reasoning quality
- **`thinking_level`** controls reasoning depth: `high` (default for Gemini 3), `medium`, `low`, `minimal` (Flash only)

In [ ]:
mn,inp = 'models/gemini-3-flash-preview',[{"role": "user", "parts": [{"text": "What is 31231231 * 12312?"}]}]
resp = await gem_cli.models.generate_content(model=mn,contents=inp,generation_config={"thinking_config": {"include_thoughts": True}})
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, generation_config={"thinking_config": {"include_thoughts": True}}, stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

🧠

🧠

31,231,23

1 × 12,312 = **384,518,916,0

72**

In [ ]:
L(comp.message.content).attrgot('type'), #comp.message.content

([<PartType.thinking: 'thinking'>, <PartType.text: 'text'>],)

In [ ]:
L(o.message.content).attrgot('type')#, o.message.content

[<PartType.thinking: 'thinking'>, <PartType.text: 'text'>]

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
L(comp.message.content).attrgot('type'), L(o.message.content).attrgot('type')

([<PartType.thinking: 'thinking'>, <PartType.text: 'text'>],
 [<PartType.thinking: 'thinking'>, <PartType.text: 'text'>])

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=20, completion_tokens=36, total_tokens=1504, raw={'promptTokenCount': 20, 'candidatesTokenCount': 36, 'totalTokenCount': 1504, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 1448}),
 Usage(prompt_tokens=20, completion_tokens=36, total_tokens=1275, raw={'promptTokenCount': 20, 'candidatesTokenCount': 36, 'totalTokenCount': 1275, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 1219}))

#### Tool call

In [ ]:
inp = [{"role": "user", "parts": [{"text": 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]
resp = await gem_cli.models.generate_content(model=mn, contents=inp, tools=[{"functionDeclarations": [sch['function']]}])
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, tools=[{"functionDeclarations": [sch['function']]}], stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

In [ ]:
comp.tool_calls

[ToolCall(id='nhn6m6n5', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'thoughtSignature': 'EoIDCv8CAQw51sem1fSURyT8K5OgvfiQZ4zOOXCSWlS02xiEI62r/40C0p3ns7FKDa3vxB8BxdceA8tmpbUyqEibihrx6i4RPS7CqWoPgQi0xz1K+A7FlX+SnVCBmu935QoHelOEqrbfT8VoyqT9HGSYG5fj3z20MJzXxMpu0q8SrxGeiNufN/zjKWv6JOtREg/ShLoMBAOdxKdWeMqQdGxlBrijyiltgQB60EysK4ClmhBAxjd33sNoNnqm1+u5/IOpyFZSQjMUNaUZfizMii2PYS9qATIXbKVSDOVQQo6lz+ll4JnDg+Mxc+Z62KxXAIRm6oc9lAY47rUE+ajvyBE7N9LHRKVZ5/dCxCeaw4t+665e9VuaPxt/Ies3/xiHkkkdqkBqzS+8cYeGikyqUi94IMV3yBacMu7jFRjXUTJoePiE6fQ3xQ0/Ebe9GKFNr7+frA9BuUh59GdQDfFs2KvbV65QOzvkq/Xtdu1oyuxmHvo+TlcKP6grzMxVORv7yVxIjjo='}),
 ToolCall(id='v0s1jgdn', name='simple_add', arguments={'b': 20, 'a': 10}, server=False, extra={})]

In [ ]:
o.tool_calls

[ToolCall(id='92y7j63g', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'thoughtSignature': 'Eq0CCqoCAQw51scUVe0gNq+5rlgOmGMV3IGknLAquWdie8uQToKaqjoXDJ0Izp5YpJ1OVFWSLoCElHx97Q6W/pFHvsOzRj0SwJZW2mdOnlZmkvc8Z+xgj87wDpjOs7uZDNkltHs30XrBQgzJs6Ewpk+X9JPtJGbcURv6v04vaouxqTf/1jwId+rn5gHYKgz7CgwGsY1bOv15zlR+pvd0KbCAX0XT70VMkIE37d4toP4+gQmiQEhVyRR9idSBPCwPf3FiwVxWLksKQheweWhErb1T5gklBcU8NgVlGmpeqVHH4FfrQ8pvyekel2V8TspvBsDIsALWrtWOO2C8JgvUk+G2wl4pLAEmpDbHn2zZy2LYqFzmylqTOhhoT+MK7ZaTykmi6mUSz6NJqHmyUgh2mA=='}),
 ToolCall(id='lm7axag6', name='simple_add', arguments={'b': 20, 'a': 10}, server=False, extra={})]

In [ ]:
comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'nhn6m6n5', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'thoughtSignature': 'EoIDCv8CAQw51sem1fSURyT8K5OgvfiQZ4zOOXCSWlS02xiEI62r/40C0p3ns7FKDa3vxB8BxdceA8tmpbUyqEibihrx6i4RPS7CqWoPgQi0xz1K+A7FlX+SnVCBmu935QoHelOEqrbfT8VoyqT9HGSYG5fj3z20MJzXxMpu0q8SrxGeiNufN/zjKWv6JOtREg/ShLoMBAOdxKdWeMqQdGxlBrijyiltgQB60EysK4ClmhBAxjd33sNoNnqm1+u5/IOpyFZSQjMUNaUZfizMii2PYS9qATIXbKVSDOVQQo6lz+ll4JnDg+Mxc+Z62KxXAIRm6oc9lAY47rUE+ajvyBE7N9LHRKVZ5/dCxCeaw4t+665e9VuaPxt/Ies3/xiHkkkdqkBqzS+8cYeGikyqUi94IMV3yBacMu7jFRjXUTJoePiE6fQ3xQ0/Ebe9GKFNr7+frA9BuUh59GdQDfFs2KvbV65QOzvkq/Xtdu1oyuxmHvo+TlcKP6grzMxVORv7yVxIjjo='}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'v0s1jgdn', 'name': 'simple_add', 'arguments': {'b': 20, 'a': 10}, 'server': False})]

In [ ]:
o.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': '92y7j63g', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'thoughtSignature': 'Eq0CCqoCAQw51scUVe0gNq+5rlgOmGMV3IGknLAquWdie8uQToKaqjoXDJ0Izp5YpJ1OVFWSLoCElHx97Q6W/pFHvsOzRj0SwJZW2mdOnlZmkvc8Z+xgj87wDpjOs7uZDNkltHs30XrBQgzJs6Ewpk+X9JPtJGbcURv6v04vaouxqTf/1jwId+rn5gHYKgz7CgwGsY1bOv15zlR+pvd0KbCAX0XT70VMkIE37d4toP4+gQmiQEhVyRR9idSBPCwPf3FiwVxWLksKQheweWhErb1T5gklBcU8NgVlGmpeqVHH4FfrQ8pvyekel2V8TspvBsDIsALWrtWOO2C8JgvUk+G2wl4pLAEmpDbHn2zZy2LYqFzmylqTOhhoT+MK7ZaTykmi6mUSz6NJqHmyUgh2mA=='}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'lm7axag6', 'name': 'simple_add', 'arguments': {'b': 20, 'a': 10}, 'server': False})]

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(set(comp.message.content[0].data.keys()), set(o.message.content[0].data.keys()))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=77, completion_tokens=38, total_tokens=221, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 221, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 106}),
 Usage(prompt_tokens=77, completion_tokens=38, total_tokens=173, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 173, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 58}))

#### Web search

Gemini's Google Search is a transparent server-side tool — unlike Anthropic/OpenAI, it produces no `functionCall` parts or tool_use blocks. Search results appear as `groundingMetadata` on the candidate, detected by `usage_from_gemini` to set `server_tool_use: {"google_search": 1}`. The response content is plain text only, so `tool_calls` is always `[]`.

In [ ]:
inp = [{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}]
resp = await gem_cli.models.generate_content(model=mn, contents=inp, tools=[{"googleSearch": {}}])
comp = normalize_gemini_generate(resp, mn)

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, tools=[{"googleSearch": {}}], stream=True, _query={"alt": "sse"})
async for o in _acollect_stream_gemini(norm_and_yield(resp, normalize_gemini_event)): print_stream(o)

Today in Istanbul, it

 is a clear evening with a temperature of **53°F (12°C)**.

Earlier in the day, the weather

 was mostly sunny with a high of approximately **57°F (14°C)**. The forecast for the remainder

 of tonight remains clear, with temperatures expected to drop to a low of around **47°F (8°C)**.



**Weather Details for Today (April 23, 2026):**
*   **Conditions:**

 Sunny day / Clear night
*   **High Temperature:** 57°F (14°C)
*   **Low Temperature

:** 47°F (8°C)
*   **Humidity:** 61%
*   

**Chance of Rain:** Minimal (0-20%)

If you are planning for tomorrow, Friday, April 

24, expect slightly warmer weather with mostly sunny skies and a high of **61°F (16°C)**.

In [ ]:
len(comp.message.content)

1

In [ ]:
t,c = comp.message.content[0].text, comp.message.content[0].data['citations']

In [ ]:
def add_gem_links(t, c):
    for s in c['groundingSupports']:
        links = ', '.join(f'<a href="{c["groundingChunks"][i]["web"]["uri"]}">{i+1}</a>' for i in s['groundingChunkIndices'])
        t = t.replace(s['segment']['text'], f"{s['segment']['text']} {links}", 1)
    return t

In [ ]:
Markdown(add_gem_links(t,c))

<div class="prose">

Today, Friday, April 24, 2026, the weather in Istanbul is primarily **sunny** and mild. <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">1</a> 

Here are the key details for today's forecast:
*   **Temperature:** Current temperature is approximately **16°C (60°F)**. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGgZ6wCy21M7drEa4kR9JhQkcWEHGa3pVi-kxUWb0XMOvUzr0NL28TD3yNPo9JvAYkq_QlgVtFtElknGliaKIdeY3Yku1pilOJE5E9MI-H_9Q05SzgX5CyS63wm-AbzaR6eonGQMuJ9D2nJN37g0Pq4zifFI06ilOzP">2</a>, <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">1</a> The daytime high is expected to reach **16°C (61°F)**, while the low tonight will drop to around **10°C (50°F)**. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGgZ6wCy21M7drEa4kR9JhQkcWEHGa3pVi-kxUWb0XMOvUzr0NL28TD3yNPo9JvAYkq_QlgVtFtElknGliaKIdeY3Yku1pilOJE5E9MI-H_9Q05SzgX5CyS63wm-AbzaR6eonGQMuJ9D2nJN37g0Pq4zifFI06ilOzP">2</a>
*   **Conditions:** It is sunny throughout the day, with a few clouds potentially appearing later in the evening. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGgZ6wCy21M7drEa4kR9JhQkcWEHGa3pVi-kxUWb0XMOvUzr0NL28TD3yNPo9JvAYkq_QlgVtFtElknGliaKIdeY3Yku1pilOJE5E9MI-H_9Q05SzgX5CyS63wm-AbzaR6eonGQMuJ9D2nJN37g0Pq4zifFI06ilOzP">2</a>, <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">1</a>
*   **Precipitation:** There is a very low chance of rain (around 10% during the day and 0% currently). <a href="https://www.google.com/search?q=weather+in+Istanbul,+TR">1</a>, <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGgZ6wCy21M7drEa4kR9JhQkcWEHGa3pVi-kxUWb0XMOvUzr0NL28TD3yNPo9JvAYkq_QlgVtFtElknGliaKIdeY3Yku1pilOJE5E9MI-H_9Q05SzgX5CyS63wm-AbzaR6eonGQMuJ9D2nJN37g0Pq4zifFI06ilOzP">2</a>
*   **Humidity:** Humidity levels are around **56% to 63%**, making for a relatively comfortable day. <a href="https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGgZ6wCy21M7drEa4kR9JhQkcWEHGa3pVi-kxUWb0XMOvUzr0NL28TD3yNPo9JvAYkq_QlgVtFtElknGliaKIdeY3Yku1pilOJE5E9MI-H_9Q05SzgX5CyS63wm-AbzaR6eonGQMuJ9D2nJN37g0Pq4zifFI06ilOzP">2</a>

If you are planning to be outdoors, it is a great day for it, though you might want a light jacket for the cooler evening hours.

</div>

In [ ]:
len(o.message.content)

1

In [ ]:
L(o.message.content).attrgot('type')

[<PartType.text: 'text'>]

citations are streamed in `_acollect_stream`, Gemini yields it as the last delta:

In [ ]:
# t,c = o.message.content[0].text, o.message.content[0].data['citations'][0]

In [ ]:
# Markdown(add_gem_links(t,c))

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)

In [ ]:
comp.usage, o.usage

(Usage(prompt_tokens=73, completion_tokens=213, total_tokens=941, raw={'promptTokenCount': 73, 'candidatesTokenCount': 213, 'totalTokenCount': 941, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 73}], 'thoughtsTokenCount': 655, 'server_tool_use': {'google_search': 1}}),
 Usage(prompt_tokens=48, completion_tokens=213, total_tokens=559, raw={'promptTokenCount': 48, 'candidatesTokenCount': 213, 'totalTokenCount': 559, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 48}], 'thoughtsTokenCount': 298, 'server_tool_use': {'google_search': 1}}))

### acollect_stream

In [ ]:
#| export
async def acollect_stream(it, model=None, api_name=None, vendor_name=None):
    _norm = {ApiName.openai: normalize_openai_response_event, ApiName.openai_chat: normalize_openai_chat_delta,
             ApiName.anthropic: normalize_anthropic_event, ApiName.gemini: normalize_gemini_event}
    _coll = {ApiName.openai: _acollect_stream_openai_responses, ApiName.openai_chat: _acollect_stream_openai_chat,
             ApiName.anthropic: _acollect_stream_anthropic, ApiName.gemini: _acollect_stream_gemini}
    if api_name not in _norm: raise ValueError(f"Unknown api_name: {api_name}")
    async for o in _coll[api_name](norm_and_yield(it, _norm[api_name]), model=model, vendor_name=vendor_name): yield o

In [ ]:
# OpenAI Responses
mn, inp = 'gpt-4o-mini', 'Say hi'
resp = await oai_cli.responses.create_response(model=mn, input=inp, stream=True)
async for o in acollect_stream(resp, model=mn, api_name=ApiName.openai): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi

 there

!

 How

 can

 I

 help

 you

 today

?

In [ ]:
# OpenAI Chat
mn, msgs = 'gpt-4o-mini', [{"role": "user", "content": "Say hi"}]
resp = await oai_cli.chat.create_chat_completion(model=mn, messages=msgs, stream=True, stream_options={"include_usage": True})
async for o in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi

 there

!

 How

 can

 I

 assist

 you

 today

?

In [ ]:
# Anthropic
mn, msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "Say hi"}]
hdrs = {"anthropic-version": "2023-06-01"}
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, max_tokens=100, headers=hdrs, stream=True)
async for o in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi! How are

 you doing today?

In [ ]:
# Gemini
mn, inp = 'models/gemini-3-flash-preview', [{"role": "user", "parts": [{"text": "Say hi"}]}]
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp, model=mn, api_name=ApiName.gemini): print_stream(o)
test_eq(o.message.content[0].type, PartType.text)
test_eq(o.finish_reason, 'stop')

Hi! How can I help you today?

In [ ]:
o.api_name, o.vendor_name

(<api_name.gemini: 'gemini'>, None)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()